# Phase 7 — Persistent Memory & Session Isolation (Bonus)

> **Gemini-only project.** Every cell here uses Google Gemini (chat + embeddings) via `langchain-google-genai`. The only key the core needs is `GOOGLE_API_KEY`. This notebook is a **scaffold** — run it top-to-bottom *after* Claude Code finishes this phase. If a cell references a module that doesn't exist yet, that phase hasn't been built.

**Goal:** memory recalls across sessions; access-control blocks cross-client reads.

**Key idea:** `thread_id = f'{client_id}-{session_id}'` IS the privacy boundary.


In [ ]:
# --- Setup: load .env and make the project importable ---
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # repo root
from dotenv import load_dotenv
load_dotenv('../.env', override=True)  # override=True: beat VS Code's stale cached env vars

# This project is Gemini-only. Confirm the one required key is present.
assert os.getenv('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY in .env (see 00_prerequisites_and_setup.md)'
print('GOOGLE_API_KEY loaded:', bool(os.getenv('GOOGLE_API_KEY')))
print('SEC_USER_AGENT   :', os.getenv('SEC_USER_AGENT', '(set before Phase 5)'))

## 1. Two sessions for CLT-001 — second recalls the first

In [ ]:
from app.graph.builder import GraphBuilder
graph = GraphBuilder().with_all().build()  # compiled with the SqliteSaver checkpointer
cfg1 = {'configurable': {'thread_id': 'CLT-001-sess-1'}}
graph.invoke({'messages': [('user', 'What do I own?')], 'client_id': 'CLT-001',
              'session_id': 'sess-1'}, cfg1)
cfg2 = {'configurable': {'thread_id': 'CLT-001-sess-2'}}
out = graph.invoke({'messages': [('user', 'What did we discuss last time?')],
                    'client_id': 'CLT-001', 'session_id': 'sess-2'}, cfg2)
print(out['messages'][-1].content)

## 2. Access control blocks cross-client reads (Interceptor)

In [ ]:
from app.guardrails.access_control import verify_client_access
try:
    verify_client_access('CLT-002', 'CLT-001')  # CLT-002 session reaching for CLT-001
except PermissionError as e:
    print('Blocked as expected:', e)

## ✅ Acceptance check

In [ ]:
import pytest
try:
    verify_client_access('CLT-002', 'CLT-001'); print('FAIL - should have raised')
except PermissionError:
    print('Phase 7 acceptance: PASS (isolation enforced)')